In [4]:
%pip install opencv-python-headless

   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
   - -------------------------------------- 1.3/40.1 MB 8.8 MB/s eta 0:00:05
   ---- ----------------------------------- 4.2/40.1 MB 12.3 MB/s eta 0:00:03
   --------- ------------------------------ 9.4/40.1 MB 17.5 MB/s eta 0:00:02
   ------------------ --------------------- 18.9/40.1 MB 25.5 MB/s eta 0:00:01
   ----------------------------------- ---- 35.7/40.1 MB 38.0 MB/s eta 0:00:01
   ---------------------------------------- 40.1/40.1 MB 39.7 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


# RT-DETRv2 Fine-tuned — Video Inference

Load the fine-tuned RT-DETRv2 model and run frame-by-frame inference on a fog video, then write an annotated output video.

In [5]:
import torch
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import AutoImageProcessor, RTDetrV2ForObjectDetection

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

f:\repos\COMP9130proj\COMP9130_Final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [6]:
# Load the fine-tuned RT-DETRv2 model from the local checkpoint
MODEL_DIR = Path("rtdetrv2_finetuned")   # 100 % fine-tuned checkpoint

processor = AutoImageProcessor.from_pretrained(str(MODEL_DIR))
model = RTDetrV2ForObjectDetection.from_pretrained(str(MODEL_DIR)).to(device).eval()

id2label = model.config.id2label
print(f"Loaded model from: {MODEL_DIR}")
print(f"Classes ({len(id2label)}): {list(id2label.values())}")

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 6769.03it/s]


Loaded model from: rtdetrv2_finetuned
Classes (80): ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [7]:
# ── Config ─────────────────────────────────────────────────────────────────
VIDEO_IN   = Path("videos/fog1.mp4")
VIDEO_OUT  = Path("videos/fog1_rtdetrv2_annotated.mp4")
SCORE_THR  = 0.35          # detection confidence threshold

# Colour palette (BGR) – rotates through labels
_PALETTE = [
    (0, 255, 0),    # lime
    (0, 200, 255),  # yellow-ish
    (255, 100, 0),  # blue
    (0, 120, 255),  # orange
    (200, 0, 255),  # magenta
]

def label_color(label_id: int):
    return _PALETTE[label_id % len(_PALETTE)]


# ── Open input video ────────────────────────────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_IN))
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video: {VIDEO_IN}")

src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
src_w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Input  : {VIDEO_IN}  |  {src_w}x{src_h} @ {src_fps:.1f} fps  |  {n_frames} frames")

# ── Prepare output writer ───────────────────────────────────────────────────
VIDEO_OUT.parent.mkdir(parents=True, exist_ok=True)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(VIDEO_OUT), fourcc, src_fps, (src_w, src_h))

# ── Frame-by-frame inference ────────────────────────────────────────────────
frame_idx = 0
with torch.no_grad():
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        # Convert BGR → PIL RGB for the processor
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)

        inputs = processor(images=pil_image, return_tensors="pt").to(device)
        outputs = model(**inputs)

        target_sizes = torch.tensor([[src_h, src_w]], device=device)
        results = processor.post_process_object_detection(
            outputs, threshold=SCORE_THR, target_sizes=target_sizes
        )[0]

        # Draw detections onto the BGR frame
        for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
            x1, y1, x2, y2 = [int(v) for v in box.tolist()]
            lid = int(label)
            color = label_color(lid)
            cls_name = id2label.get(lid, str(lid))
            text = f"{cls_name} {float(score):.2f}"

            cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color, 2)

            (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            ty = max(y1 - 4, th + baseline)
            cv2.rectangle(frame_bgr, (x1, ty - th - baseline), (x1 + tw, ty + baseline), color, cv2.FILLED)
            cv2.putText(frame_bgr, text, (x1, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1, cv2.LINE_AA)

        writer.write(frame_bgr)
        frame_idx += 1
        if frame_idx % 30 == 0:
            print(f"  processed {frame_idx}/{n_frames} frames ...", end="\r")

cap.release()
writer.release()
print(f"\nDone. Annotated video saved to: {VIDEO_OUT}")
print(f"Total frames processed: {frame_idx}")

Input  : videos\fog1.mp4  |  1920x1080 @ 30.0 fps  |  195 frames
  processed 180/195 frames ...
Done. Annotated video saved to: videos\fog1_rtdetrv2_annotated.mp4
Total frames processed: 195
